# Proyecto Simula

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import random
import os
import time
from scipy.stats import linregress
import time
from scipy.stats import linregress
from scipy.special import expit
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from IPython.display import display,Markdown

### Funciones de Energía

In [ ]:
def Energia(a,X,i,j):
    N = np.size(X,1)
    energia = 0
    if i>0:
        energia += 0.5*(X[i,j]+a[i,j]-X[i-1,j]-a[i-1,j])**2
    if i<N-1:
        energia += 0.5*(X[i,j]+a[i,j]-X[i+1,j]-a[i+1,j])**2
    if j>0:
        energia += 0.5*(X[i,j]+a[i,j]-X[i,j-1]-a[i,j-1])**2
    if j<N-1:
        energia += 0.5*(X[i,j]+a[i,j]-X[i,j+1]-a[i,j+1])**2
    return energia

def EnergiaTotal(a,phi):
    X = a+phi
    energia = 0
    energia += 0.5*np.sum((X[1:,:]-X[:-1,:])**2)
    energia += 0.5*np.sum((X[:,1:]-X[:,:-1])**2)
    return energia

# Versión para Optimizar

def DeltaEnergia(a,phi,i,j,u):
    N = np.size(phi,1)
    Xij = a[i,j]+phi[i,j]
    vecinos = 0
    cantidad = 0

    if i>0:
        vecinos += a[i-1,j]+phi[i-1,j]
        cantidad += 1
    if i<N-1:
        vecinos += a[i+1,j]+phi[i+1,j]
        cantidad += 1
    if j>0:
        vecinos += a[i,j-1]+phi[i,j-1]
        cantidad += 1
    if j<N-1:
        vecinos += a[i,j+1]+phi[i,j+1]
        cantidad += 1

    return u*(cantidad*Xij-vecinos)+0.5*cantidad*u**2

## Ising para $\beta$ fijo

In [ ]:
# expit(a) = 1/ (1 + exp(a)), para que no haya overflow de la exponencial manual.

def IsingGlauber(N,beta,a,n):
    # Fijamos a en cero en los bordes
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)
    # Energía inicial
    energias = [EnergiaTotal(a,phi)]
    # Lista con la variación de los betas (fijo en este caso)
    historial_beta = [beta]

    for k in range(n):
        # Elegimos punto aleatorio uniforme (incluyendo los bordes)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        U = np.random.uniform(0,1)
        if U <= proba:
            phi[i,j] += u

        # Guardamos la energía cada n//10 iteraciones
        if (k+1)%(n//10) == 0:
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return phi,energias,historial_beta

In [ ]:
# Comprobación y tabla comparativa
N = 10
n = 10000
beta = 20
rng = np.random.default_rng(seed=217896193)
a = rng.uniform(0,1,size=(N,N))
np.random.seed(123)
phi,energias,historial_beta = IsingGlauber(N,beta,a,n)

print("Simulación para",n,"actualizaciones")
print("___________________________________________")
print("Energía Inicial:",EnergiaTotal(a,np.zeros((N,N))))
print("Energía Final:",EnergiaTotal(a,phi))
print("Mejora de Energía:",energias[0]-energias[-1])
print("___________________________________________")

iteraciones = list(range(0,n+1,n//10))
tabla = pd.DataFrame({"Iteración":iteraciones,"β":historial_beta,"Energía de Dirichlet":energias})
print("\nEvolución de la energía para beta =",beta)
print(tabla.to_string(index=False))

### Comparación energía v/s beta (N fijo)

In [ ]:
# PARÁMETROS
N = 10
n = 100000

betas = np.unique(np.concatenate([
    np.geomspace(0.00001,0.001,30),
    np.geomspace(0.001,0.01,25),
    np.linspace(0.01,0.1,30),
    np.linspace(0.1,1,40),
    np.linspace(1,5,25),
    np.linspace(5,20,31),
    np.linspace(20,50,16)
]))

rng = np.random.default_rng(seed=217896193)
a = rng.uniform(0,1,size=(N,N))
a[0,:] = 0
a[-1,:] = 0
a[:,0] = 0
a[:,-1] = 0

energias_betas = []
energias_finales = []
diferencias = []

# EJECUTAMOS GLAUBER PARA CADA BETA
for beta in betas:
    phi,energias,historial_beta = IsingGlauber(N,beta,a,n)

    energias_betas.append(energias)
    energias_finales.append(energias[-1])
    diferencias.append(energias[0]-energias[-1])

# TABLA DE ENERGÍAS FINALES Y DIFERENCIAS
tabla = pd.DataFrame({
    "β":betas,
    "Energía final":energias_finales,
    "Diferencia":diferencias
})

print("Energía final y reducción de energía para distintos β:")
print(tabla.to_string(index=False,float_format="%.6f"))

# GRÁFICO ENERGÍA VS ITERACIONES
iteraciones = np.arange(0,n+1,n//10)

plt.figure(figsize=(10,6))
for k,beta in enumerate(betas):
    plt.plot(iteraciones,energias_betas[k],alpha=0.5)

plt.yscale("log")
plt.xlabel("Iteraciones")
plt.ylabel("Energía de Dirichlet")
plt.title("Evolución de la energía para distintos β")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# GRÁFICO BETA VS ENERGÍA FINAL
plt.figure(figsize=(10,6))
plt.plot(betas,energias_finales,marker="o",markersize=3)

plt.xscale("log")
plt.xlabel("β")
plt.ylabel("Energía final de Dirichlet")
plt.title("Energía final vs β")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Encontrar el Mejor $\beta$

In [ ]:
# PARÁMETROS
Ns = list(range(10,101,10))
R = 10
Bmax = 100
rondas = 5
beta_min,beta_max = 3,50
tolerancia = 0.01
tiempo_max = 85*60

resultados = []
resumen = []
inicio = time.perf_counter()
detener = False

# BÚSQUEDA DEL BETA DE ESTABILIZACIÓN
for N in Ns:
    inicio_N = time.perf_counter()
    n = Bmax*N**2
    matrices = []

    # GENERAMOS LAS MISMAS MATRICES PARA TODOS LOS BETAS
    for r in range(R):
        a = np.random.uniform(0,1,size=(N,N))
        a[0,:] = 0
        a[-1,:] = 0
        a[:,0] = 0
        a[:,-1] = 0
        matrices.append(a)

    evaluados = {}
    inferior,superior = beta_min,beta_max

    print(f"\nN={N} | {n:,} iteraciones | {R} repeticiones",flush=True)

    for ronda in range(rondas):
        betas = np.linspace(inferior,superior,5)

        for beta in betas:
            beta = round(float(beta),8)
            if beta in evaluados:
                continue

            energias_finales = []

            # EVALUAMOS CADA BETA CON LAS MISMAS MATRICES
            for a in matrices:
                if time.perf_counter()-inicio>=tiempo_max:
                    detener = True
                    break

                phi,energias,historial_beta = IsingGlauber(N,beta,a,n)
                energias_finales.append(energias[-1])

            if detener:
                break

            promedio = np.mean(energias_finales)
            desviacion = np.std(energias_finales,ddof=1)
            evaluados[beta] = (promedio,desviacion)

            resultados.append({
                "N":N,"β":beta,
                "Energía promedio":promedio,
                "Desviación estándar":desviacion
            })

            # IMPRIMIMOS EL TIEMPO DESPUÉS DE EVALUAR CADA BETA
            minutos_N = (time.perf_counter()-inicio_N)/60
            minutos_total = (time.perf_counter()-inicio)/60

            print(f"  Ronda {ronda+1}/{rondas} | β={beta:.5f} | E={promedio:.4f} | Grilla: {minutos_N:.2f} min | Total: {minutos_total:.2f} min",flush=True)

        if detener or not evaluados:
            break

        # MENOR BETA DENTRO DEL 1% DEL MÍNIMO OBSERVADO
        tabla_N = pd.DataFrame([
            {"β":b,"Energía promedio":v[0]} for b,v in evaluados.items()
        ]).sort_values("β")

        energia_minima = tabla_N["Energía promedio"].min()
        umbral = energia_minima*(1+tolerancia)
        mejor_beta = tabla_N.loc[
            tabla_N["Energía promedio"]<=umbral,"β"
        ].min()

        print(f"  Mejor β ronda {ronda+1}: {mejor_beta:.5f} | E mínima={energia_minima:.4f}",flush=True)

        # REFINAMOS EL INTERVALO
        valores = np.sort(np.unique(np.append(betas,mejor_beta)))
        posicion = np.argmin(np.abs(valores-mejor_beta))
        inferior = max(beta_min,valores[max(0,posicion-1)])
        superior = min(beta_max,valores[min(len(valores)-1,posicion+1)])

        if superior-inferior<1e-6:
            break

    # GUARDAMOS LOS RESULTADOS
    if not detener and evaluados:
        resumen.append({
            "N":N,
            "β estabilización":mejor_beta,
            "Energía mínima":energia_minima,
            "Umbral 1%":umbral,
            "Barridos":Bmax
        })

        minutos_N = (time.perf_counter()-inicio_N)/60
        minutos_total = (time.perf_counter()-inicio)/60

        print(f"FINALIZADO N={N} | β={mejor_beta:.5f} | Grilla: {minutos_N:.2f} min | Total: {minutos_total:.2f} min",flush=True)

    if detener:
        print("\nLímite de tiempo alcanzado.")
        break

# TABLAS
tabla = pd.DataFrame(resultados)
tabla_resumen = pd.DataFrame(resumen)

print("\nRESULTADOS POR GRILLA:")
if not tabla_resumen.empty:
    print(tabla_resumen.to_string(index=False,float_format="%.5f"))
else:
    print("No se completó ninguna grilla.")

print(f"\nTiempo total: {(time.perf_counter()-inicio)/60:.2f} minutos")


# GRÁFICO 2: ENERGÍA NORMALIZADA VS BETA
if not tabla.empty:
    plt.figure(figsize=(10,6))
    for N in sorted(tabla["N"].unique()):
        datos = tabla[tabla["N"]==N].sort_values("β")
        plt.plot(
            datos["β"],
            datos["Energía promedio"]/(2*N*(N-1)),
            marker="o",label=f"N={N}"
        )
    plt.xlabel("β")
    plt.ylabel("Energía final promedio por enlace")
    plt.title("Energía normalizada vs β para distintos N")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# GRÁFICO 3: BETA DE ESTABILIZACIÓN VS N
if not tabla_resumen.empty:
    plt.figure(figsize=(9,5))
    plt.plot(tabla_resumen["N"],tabla_resumen["β estabilización"],marker="o")
    plt.xlabel("N")
    plt.ylabel("β de estabilización")
    plt.title("β de estabilización observado para cada grilla")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Ising para $\beta$ variante

### Variación Lineal      
$$
\beta_k = \beta_0 + \frac{k}{n-1}(\beta_f - \beta_0)
$$

In [ ]:
def IsingGlauberLineal(N,beta0,betaf,a,n):
    # Fijamos a en cero en los bordes
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)
    # Energía inicial
    energias = [EnergiaTotal(a,phi)]
    # Lista con la variación de los betas
    historial_beta = [beta0]

    for k in range(n):
        # Aumentamos beta linealmente en cada iteración
        beta = beta0+(betaf-beta0)*k/(n-1)

        # Elegimos punto aleatorio uniforme (incluyendo los bordes)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        U = np.random.uniform(0,1)
        if U <= proba:
            phi[i,j] += u

        # Guardamos la energía y beta cada n//10 iteraciones
        if (k+1)%(n//10) == 0:
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return phi,energias,historial_beta

### Variación Cuadrática
$$
\beta_k = \beta_0 + \left(\beta_f - \beta_0\right)\left(\frac{k}{n-1}\right)^2
$$

In [ ]:
def IsingGlauberCuadratico(N,beta0,betaf,a,n):
    # Fijamos a en cero en los bordes
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)
    # Energía inicial
    energias = [EnergiaTotal(a,phi)]
    # Lista con la variación de los betas
    historial_beta = [beta0]

    for k in range(n):
        # Aumentamos beta cuadráticamente en cada iteración
        beta = beta0+(betaf-beta0)*(k/(n-1))**2

        # Elegimos punto aleatorio uniforme (incluyendo los bordes)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        U = np.random.uniform(0,1)
        if U <= proba:
            phi[i,j] += u

        # Guardamos la energía y beta cada n//10 iteraciones
        if (k+1)%(n//10) == 0:
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return phi,energias,historial_beta

### Variación Geométrica
$$
\beta_k = \beta_0\left(\frac{\beta_f}{\beta_0}\right)^\frac{k}{n-1}
$$

In [ ]:
def IsingGlauberGeometrico(N,beta0,betaf,a,n):
    # Fijamos a en cero en los bordes
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)
    # Energía inicial
    energias = [EnergiaTotal(a,phi)]
    # Lista con la variación de los betas
    historial_beta = [beta0]

    for k in range(n):
        # Aumentamos beta geométricamente en cada iteración
        beta = beta0*(betaf/beta0)**(k/(n-1))

        # Elegimos punto aleatorio uniforme (incluyendo los bordes)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        U = np.random.uniform(0,1)
        if U <= proba:
            phi[i,j] += u

        # Guardamos la energía y beta cada n//10 iteraciones
        if (k+1)%(n//10) == 0:
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return phi,energias,historial_beta

### Variación Subida-Bajada geométrica

In [ ]:
def IsingGlauberAdaptativo(N,beta0,betamin,betaf,a,n):
    # Fijamos a en cero en los bordes
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)
    # Energía inicial
    energias = [EnergiaTotal(a,phi)]
    # Lista con la variación de los betas
    historial_beta = [beta0]

    # Definimos calentamiento (20%) y enfriamiento (80%)
    calentamiento = int(0.1*n)

    for k in range(n):
        # Calentamiento: disminuimos beta geométricamente
        if k < calentamiento:
            beta = beta0*(betamin/beta0)**(k/(calentamiento-1))

        # Enfriamiento: aumentamos beta geométricamente
        else:
            beta = betamin*(betaf/betamin)**((k-calentamiento)/(n-calentamiento-1))

        # Elegimos punto aleatorio uniforme (incluyendo los bordes)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        U = np.random.uniform(0,1)
        if U <= proba:
            phi[i,j] += u

        # Guardamos la energía y beta cada n//10 iteraciones
        if (k+1)%(n//10) == 0:
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return phi,energias,historial_beta

In [ ]:
# PARÁMETROS
N = 20
Bmax = 100
n = Bmax*N**2
beta0,betamin,betaf,beta_fijo = 1,0.5,20,11

# ITERACIONES
iteraciones = np.linspace(0,n,500)
t = iteraciones/n

# CINCO ESQUEMAS DE TEMPERATURA
fijo = np.full_like(t,beta_fijo)
lineal = beta0+(betaf-beta0)*t
cuadratico = beta0+(betaf-beta0)*t**2
geometrico = beta0*(betaf/beta0)**t
adaptativo = np.where(t<0.1,beta0*(betamin/beta0)**(t/0.1),
                      betamin*(betaf/betamin)**((t-0.1)/0.9))

# GRÁFICO BETA VS ITERACIONES
plt.figure(figsize=(9,5))
plt.plot(iteraciones,fijo,label="Fijo",color="black")
plt.plot(iteraciones,lineal,label="Lineal",color="blue")
plt.plot(iteraciones,cuadratico,label="Cuadrático",color="orange")
plt.plot(iteraciones,geometrico,label="Geométrico",color="green")
plt.plot(iteraciones,adaptativo,label="Calent.-enfr.",color="red")

plt.xlabel("Iteraciones")
plt.ylabel(r"$\beta_k$")
plt.title(r"Evolución de $\beta$ según el método ($N=20$)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("beta_iteraciones.png",dpi=300,bbox_inches="tight")
plt.show()

### Comparación de Variaciones para distintos N

In [ ]:
# PARÁMETROS
Ns = [10,20,30,40,50]
R = 20
Bmax = 100
beta0,betamin,betaf,beta_fijo = 1,0.5,20,11

metodos = ["Fijo","Lineal","Cuadrático","Geométrico","Calentamiento-Enfriamiento"]
comparacion = []
evoluciones = {}
inicio = time.perf_counter()

# EJECUTAMOS EL EXPERIMENTO
for N in Ns:
    inicio_N = time.perf_counter()
    n = Bmax*N**2
    enlaces = 2*N*(N-1)
    print(f"\nSIMULANDO N = {N} | {n:,} iteraciones por método | {R} repeticiones",flush=True)

    for r in range(R):
        a = np.random.uniform(0,1,size=(N,N))
        a[0,:] = a[-1,:] = 0
        a[:,0] = a[:,-1] = 0

        resultados = {}
        _,resultados["Fijo"],_ = IsingGlauber(N,beta_fijo,a,n)
        _,resultados["Lineal"],_ = IsingGlauberLineal(N,beta0,betaf,a,n)
        _,resultados["Cuadrático"],_ = IsingGlauberCuadratico(N,beta0,betaf,a,n)
        _,resultados["Geométrico"],_ = IsingGlauberGeometrico(N,beta0,betaf,a,n)
        _,resultados["Calentamiento-Enfriamiento"],_ = IsingGlauberAdaptativo(N,beta0,betamin,betaf,a,n)

        for nombre,energias in resultados.items():
            E = np.array(energias)/enlaces
            comparacion.append({
                "N":N,"Repetición":r+1,"Método":nombre,
                "Energía inicial":E[0],"Energía final":E[-1],
                "Reducción":E[0]-E[-1],
                "Reducción (%)":100*(E[0]-E[-1])/E[0]
            })
            evoluciones.setdefault((N,nombre),[]).append(E)

        # IMPRIMIMOS EL TIEMPO DESPUÉS DE CADA REPETICIÓN
        minutos_N = (time.perf_counter()-inicio_N)/60
        minutos_total = (time.perf_counter()-inicio)/60
        print(f"N={N} | Repetición {r+1}/{R} | Grilla: {minutos_N:.2f} min | Total: {minutos_total:.2f} min",flush=True)

    print(f"FINALIZADO N={N} | Tiempo grilla: {(time.perf_counter()-inicio_N)/60:.2f} min | Tiempo total: {(time.perf_counter()-inicio)/60:.2f} min",flush=True)

# TABLA RESUMEN
tabla = pd.DataFrame(comparacion)
tabla_resumen = tabla.groupby(["N","Método"]).agg(
    Energia_inicial_promedio=("Energía inicial","mean"),
    Energia_final_promedio=("Energía final","mean"),
    Desviacion_estandar=("Energía final","std"),
    Reduccion_promedio=("Reducción","mean"),
    Reduccion_porcentual=("Reducción (%)","mean")
).reset_index()

tabla_energias = tabla_resumen.pivot(index="N",columns="Método",values="Energia_final_promedio")[metodos]
tabla_reducciones = tabla_resumen.pivot(index="N",columns="Método",values="Reduccion_porcentual")[metodos]

print("\nENERGÍAS FINALES NORMALIZADAS PROMEDIO:")
print(tabla_energias.to_string(float_format="%.6f"))
print("\nREDUCCIÓN PORCENTUAL PROMEDIO:")
print(tabla_reducciones.to_string(float_format="%.2f"))
print("\nMEJOR MÉTODO PARA CADA N:")
print(tabla_energias.idxmin(axis=1).to_string())
print(f"\nTIEMPO TOTAL: {(time.perf_counter()-inicio)/60:.2f} minutos")

# GRÁFICO 1: ENERGÍA FINAL NORMALIZADA VS N
plt.figure(figsize=(10,6))
for metodo in metodos:
    datos = tabla_resumen[tabla_resumen["Método"]==metodo]
    plt.errorbar(datos["N"],datos["Energia_final_promedio"],yerr=datos["Desviacion_estandar"],
                 marker="o",capsize=3,label=metodo)
plt.xlabel("N")
plt.ylabel("Energía final por enlace")
plt.title("Energía final normalizada promedio vs N")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# GRÁFICO 2: REDUCCIÓN PORCENTUAL VS N
plt.figure(figsize=(10,6))
for metodo in metodos:
    datos = tabla_resumen[tabla_resumen["Método"]==metodo]
    plt.plot(datos["N"],datos["Reduccion_porcentual"],marker="o",label=metodo)
plt.xlabel("N")
plt.ylabel("Reducción de energía (%)")
plt.title("Reducción porcentual promedio vs N")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# GRÁFICO 3: EVOLUCIÓN PROMEDIO DE LOS CINCO MÉTODOS
N_grafico = 20
barridos = np.linspace(0,Bmax,11)
plt.figure(figsize=(10,6))
for metodo in metodos:
    curvas = np.array(evoluciones[(N_grafico,metodo)])
    promedio = np.mean(curvas,axis=0)
    desviacion = np.std(curvas,axis=0,ddof=1)
    linea, = plt.plot(barridos,promedio,marker="o",label=metodo)
    plt.fill_between(barridos,promedio-desviacion,promedio+desviacion,
                     color=linea.get_color(),alpha=0.12)
plt.xlabel("Barridos")
plt.ylabel("Energía de Dirichlet por enlace")
plt.title(f"Evolución promedio de los cinco métodos - N = {N_grafico}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Ejemplo Exacto

In [ ]:
# PARÁMETROS
N = 20

# GENERAMOS LA PIRÁMIDE ESCALONADA
i,j = np.indices((N,N))
d = np.minimum.reduce([i,j,N-1-i,N-1-j])

piramide = d/3

# CONFIGURACIÓN INICIAL
a = (d%3)/3
phi_inicial = np.zeros((N,N),dtype=int)

# CONFIGURACIÓN DE LA PIRÁMIDE
phi_optima = d//3

# CALCULAMOS LAS ENERGÍAS
energia_inicial = EnergiaTotal(a,phi_inicial)
energia_piramide = EnergiaTotal(a,phi_optima)

print("Energía inicial:",round(energia_inicial,4))
print("Energía de la pirámide:",round(energia_piramide,4))
print("Reducción de energía:",round(energia_inicial-energia_piramide,4))

# GRÁFICO DE LAS CONFIGURACIONES
fig,axs = plt.subplots(1,2,figsize=(12,5))

im1 = axs[0].imshow(a,cmap="viridis",vmin=0,vmax=piramide.max())
axs[0].set_title("Configuración inicial")
fig.colorbar(im1,ax=axs[0])

im2 = axs[1].imshow(piramide,cmap="viridis",vmin=0,vmax=piramide.max())
axs[1].set_title("Pirámide escalonada")
fig.colorbar(im2,ax=axs[1])

plt.tight_layout()
plt.show()

In [ ]:
# PARÁMETROS
Ns = list(range(10,21))
R = 20
beta = 11
Bmax = 10000

# GLAUBER FIJO HASTA ENCONTRAR EL ÓPTIMO
def IsingGlauberOptimo(N,beta,a,n,energia_optima):
    phi = np.zeros((N,N),dtype=int)
    energia_actual = EnergiaTotal(a,phi)

    if np.isclose(energia_actual,energia_optima,atol=1e-8,rtol=0):
        return True,0

    for k in range(n):
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)
        u = np.random.choice([-1,1])

        deltaH = DeltaEnergia(a,phi,i,j,u)
        proba = expit(-beta*deltaH)

        if np.random.rand()<proba:
            phi[i,j] += u
            energia_actual += deltaH

            # COMPROBAMOS SI ALCANZAMOS EL ÓPTIMO
            if energia_actual<=energia_optima+1e-8:
                if np.isclose(EnergiaTotal(a,phi),energia_optima,atol=1e-8,rtol=0):
                    return True,k+1

    return False,None


# EJECUTAMOS EL EXPERIMENTO
resultados = []

for N in Ns:
    n = Bmax*N**2

    print(f"\nSimulando N = {N}, máximo = {n} iteraciones",flush=True)

    # CONFIGURACIÓN INICIAL Y ÓPTIMO GLOBAL
    i,j = np.indices((N,N))
    d = np.minimum.reduce([i,j,N-1-i,N-1-j])

    a = (d%3)/3
    phi_optima = d//3

    energia_optima = EnergiaTotal(a,phi_optima)

    # REPETIMOS GLAUBER
    for r in range(R):
        alcanzo,iteraciones = IsingGlauberOptimo(N,beta,a,n,energia_optima)

        resultados.append({
            "N":N,
            "Repetición":r+1,
            "Alcanzó óptimo":alcanzo,
            "Iteraciones":iteraciones,
            "Barridos":iteraciones/N**2 if alcanzo else np.nan,
            "Tiempo restringido":iteraciones if alcanzo else n,
            "Barridos restringidos":iteraciones/N**2 if alcanzo else Bmax
        })

        print(f"Repetición {r+1}/{R}: éxito = {alcanzo}, iteraciones = {iteraciones}",flush=True)


# TABLA DE RESULTADOS
tabla = pd.DataFrame(resultados)

# TABLA RESUMEN
tabla_resumen = tabla.groupby("N").agg(
    Exitos=("Alcanzó óptimo","sum"),
    Iteraciones_promedio=("Iteraciones","mean"),
    Iteraciones_mediana=("Iteraciones","median"),
    Desviacion_estandar=("Iteraciones","std"),
    Barridos_promedio=("Barridos","mean"),
    Tiempo_restringido_promedio=("Tiempo restringido","mean"),
    Barridos_restringidos_promedio=("Barridos restringidos","mean")
).reset_index()

tabla_resumen["Tasa_exito"] = 100*tabla_resumen["Exitos"]/R

print("\nResumen por tamaño de grilla:")
print(tabla_resumen.to_string(index=False,float_format="%.4f"))


### Tabla 1. Tiempo de llegada al óptimo global

**Glauber fijo:** β = 11, 20 repeticiones por tamaño y un máximo de 10.000 barridos.

| N | Éxitos | Iteraciones promedio | Mediana | Desv. estándar | Barridos promedio |
|---|---:|---:|---:|---:|---:|
| 10 | 19/20 | 401.290 | 454.516 | 244.746 | 4.012,90 |
| 11 | 19/20 | 358.620 | 274.391 | 265.800 | 2.963,80 |
| 12 | 16/20 | 569.451 | 580.358 | 274.842 | 3.954,52 |
| 13 | 15/20 | 1.058.592 | 1.055.422 | 394.673 | 6.263,86 |
| 14 | 11/20 | 1.420.926 | 1.411.710 | 256.726 | 7.249,62 |
| 15 | 10/20 | 1.827.715 | 1.803.847 | 301.598 | 8.123,18 |
| 16 | 5/20 | 1.972.771 | 2.122.051 | 467.250 | 7.706,14 |
| 17 | 6/20 | 2.378.732 | 2.467.085 | 451.175 | 8.230,91 |
| 18 | 4/20 | 2.719.308 | 2.878.010 | 569.868 | 8.392,92 |
| 19 | 1/20 | 3.191.883 | 3.191.883 | — | 8.841,78 |
| 20 | 2/20 | 2.858.339 | 2.858.339 | 499.552 | 7.145,85 |

*Nota:* Los promedios, medianas y desviaciones estándar consideran únicamente las simulaciones exitosas. El guion indica que no se puede calcular la desviación estándar muestral con una sola observación.

---

### Tabla 2. Tasa de éxito y tiempo restringido

| N | Éxitos | Tasa de éxito | Tiempo restringido promedio | Barridos restringidos promedio |
|---|---:|---:|---:|---:|
| 10 | 19/20 | 95% | 431.226 | 4.312,26 |
| 11 | 19/20 | 95% | 401.189 | 3.315,61 |
| 12 | 16/20 | 80% | 743.561 | 5.163,61 |
| 13 | 15/20 | 75% | 1.216.444 | 7.197,89 |
| 14 | 11/20 | 55% | 1.663.509 | 8.487,29 |
| 15 | 10/20 | 50% | 2.038.857 | 9.061,59 |
| 16 | 5/20 | 25% | 2.413.193 | 9.426,53 |
| 17 | 6/20 | 30% | 2.736.620 | 9.469,27 |
| 18 | 4/20 | 20% | 3.135.862 | 9.678,58 |
| 19 | 1/20 | 5% | 3.589.094 | 9.942,09 |
| 20 | 2/20 | 10% | 3.885.834 | 9.714,58 |

*Nota:* El tiempo restringido promedio considera tanto los éxitos como las simulaciones que no alcanzaron el óptimo, asignando a estas últimas el máximo de 10.000 barridos.


## Visualizaciones en 3D

### Modificamos las funciones para que guarden frames

In [ ]:
def IsingGlauberAnimacion(N,beta,a,n,frames=50):
    # Fijamos a en cero en los bordes (a es fijo; phi puede tomar cualquier valor entero)
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)

    # Guardamos la configuración y energía inicial
    configuraciones = [phi.copy()]
    iteraciones = [0]
    energias = [EnergiaTotal(a,phi)]

    intervalo = max(1,n//frames)

    for k in range(n):
        # Elegimos punto aleatorio uniforme (incluyendo los bordes: phi es libre en todos los sitios)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Energía actual
        H = Energia(a,phi,i,j)
        
        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        if np.random.uniform(0,1) <= proba:
            phi[i,j] += u

        # Guardamos los resultados
        if (k+1)%intervalo == 0 or k == n-1:
            configuraciones.append(phi.copy())
            iteraciones.append(k+1)
            energias.append(EnergiaTotal(a,phi))

    return configuraciones,iteraciones,energias

In [ ]:
def IsingGlauberLinealAnimacion(N,beta0,betaf,a,n,frames=60):
    # Fijamos a en cero en los bordes (a es fijo; phi puede tomar cualquier valor entero)
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)

    # Guardamos configuración, energía y beta inicial
    configuraciones = [phi.copy()]
    iteraciones = [0]
    energias = [EnergiaTotal(a,phi)]
    historial_beta = [beta0]

    intervalo = max(1,n//frames)

    for k in range(n):
        # Aumentamos beta linealmente
        beta = beta0+(betaf-beta0)*k/(n-1)

        # Elegimos punto aleatorio uniforme (incluyendo los bordes: phi es libre en todos los sitios)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Energía actual
        H = Energia(a,phi,i,j)
        
        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        if np.random.uniform(0,1) <= proba:
            phi[i,j] += u

        # Guardamos las configuraciones intermedias
        if (k+1)%intervalo == 0 or k == n-1:
            configuraciones.append(phi.copy())
            iteraciones.append(k+1)
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return configuraciones,iteraciones,energias,historial_beta

In [ ]:
def IsingGlauberCuadraticoAnimacion(N,beta0,betaf,a,n,frames=60):
    # Fijamos a en cero en los bordes (a es fijo; phi puede tomar cualquier valor entero)
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)

    # Guardamos configuración, energía y beta inicial
    configuraciones = [phi.copy()]
    iteraciones = [0]
    energias = [EnergiaTotal(a,phi)]
    historial_beta = [beta0]

    intervalo = max(1,n//frames)

    for k in range(n):
        # Aumentamos beta cuadráticamente
        beta = beta0+(betaf-beta0)*(k/(n-1))**2

        # Elegimos punto aleatorio uniforme (incluyendo los bordes: phi es libre en todos los sitios)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Energía actual
        H = Energia(a,phi,i,j)
        
        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        if np.random.uniform(0,1) <= proba:
            phi[i,j] += u

        # Guardamos las configuraciones intermedias
        if (k+1)%intervalo == 0 or k == n-1:
            configuraciones.append(phi.copy())
            iteraciones.append(k+1)
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return configuraciones,iteraciones,energias,historial_beta

In [ ]:
def IsingGlauberGeometricoAnimacion(N,beta0,betaf,a,n,frames=60):
    # Fijamos a en cero en los bordes (a es fijo; phi puede tomar cualquier valor entero)
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)

    # Guardamos configuración, energía y beta inicial
    configuraciones = [phi.copy()]
    iteraciones = [0]
    energias = [EnergiaTotal(a,phi)]
    historial_beta = [beta0]

    intervalo = max(1,n//frames)

    for k in range(n):
        # Aumentamos beta geométricamente
        beta = beta0*(betaf/beta0)**(k/(n-1))

        # Elegimos punto aleatorio uniforme (incluyendo los bordes: phi es libre en todos los sitios)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Energía actual
        H = Energia(a,phi,i,j)
        
        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        if np.random.uniform(0,1) <= proba:
            phi[i,j] += u

        # Guardamos las configuraciones intermedias
        if (k+1)%intervalo == 0 or k == n-1:
            configuraciones.append(phi.copy())
            iteraciones.append(k+1)
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return configuraciones,iteraciones,energias,historial_beta

In [ ]:
def IsingGlauberAdaptativoAnimacion(N,beta0,betamin,betaf,a,n,frames=60):
    # Fijamos a en cero en los bordes (a es fijo; phi puede tomar cualquier valor entero)
    a = a.copy()
    a[0,:] = 0
    a[-1,:] = 0
    a[:,0] = 0
    a[:,-1] = 0

    # Inicializamos phi en cero
    phi = np.zeros((N,N),dtype=int)

    # Guardamos configuración, energía y beta inicial
    configuraciones = [phi.copy()]
    iteraciones = [0]
    energias = [EnergiaTotal(a,phi)]
    historial_beta = [beta0]

    calentamiento = int(0.2*n)
    intervalo = max(1,n//frames)

    for k in range(n):
        # Calentamiento: disminuimos beta geométricamente
        if k < calentamiento:
            beta = beta0*(betamin/beta0)**(k/(calentamiento-1))

        # Enfriamiento: aumentamos beta geométricamente
        else:
            beta = betamin*(betaf/betamin)**((k-calentamiento)/(n-calentamiento-1))

        # Elegimos punto aleatorio uniforme (incluyendo los bordes: phi es libre en todos los sitios)
        i = np.random.randint(0,N)
        j = np.random.randint(0,N)

        # Energía actual
        H = Energia(a,phi,i,j)
        
        # Proponemos la actualización
        u = np.random.choice([-1,1])
        deltaH = DeltaEnergia(a,phi,i,j,u)

        # Probabilidad de aceptación según Glauber
        proba = expit(-beta*deltaH)
        if np.random.uniform(0,1) <= proba:
            phi[i,j] += u

        # Guardamos las configuraciones intermedias
        if (k+1)%intervalo == 0 or k == n-1:
            configuraciones.append(phi.copy())
            iteraciones.append(k+1)
            energias.append(EnergiaTotal(a,phi))
            historial_beta.append(beta)

    return configuraciones,iteraciones,energias,historial_beta

### Animaciones de los distintos Glauber

In [ ]:
# PARÁMETROS
N = 20
n = 100000
beta0 = 1
betamin = 0.5
betaf = 30
frames = 60

rng = np.random.default_rng(seed=42)
a = rng.uniform(0,1,size=(N,N))
a[0,:] = 0
a[-1,:] = 0
a[:,0] = 0
a[:,-1] = 0

# EJECUTAMOS LOS CINCO MODELOS
resultados = {}

resultados["Fijo"] = IsingGlauberAnimacion(N,14,a,n,frames)

resultados["Lineal"] = IsingGlauberLinealAnimacion(N,beta0,betaf,a,n,frames)

resultados["Cuadrático"] = IsingGlauberCuadraticoAnimacion(N,beta0,betaf,a,n,frames)

resultados["Geométrico"] = IsingGlauberGeometricoAnimacion(N,beta0,betaf,a,n,frames)

resultados["Calentamiento-Enfriamiento"] = IsingGlauberAdaptativoAnimacion(N,beta0,betamin,betaf,a,n,frames)

# AGREGAMOS EL HISTORIAL DE BETA AL MODELO FIJO
configuraciones,iteraciones,energias = resultados["Fijo"]
resultados["Fijo"] = (configuraciones,iteraciones,energias,[betaf]*len(iteraciones))

# ESCALA COMÚN PARA LOS CINCO MODELOS
superficies_total = [a+phi for configuraciones,_,_,_ in resultados.values() for phi in configuraciones]

zmin = min(np.min(X) for X in superficies_total)-0.3
zmax = max(np.max(X) for X in superficies_total)+0.3
norm = plt.Normalize(vmin=zmin,vmax=zmax)
cmap = plt.cm.turbo

# POSICIONES DE LOS BLOQUES
i,j = np.indices((N,N))
dx = 0.9
dy = 0.9
grosor = 0.12

# FUNCIÓN PARA CREAR CADA ANIMACIÓN
def CrearAnimacion(nombre,configuraciones,iteraciones,energias,historial_beta):
    superficies = [a+phi for phi in configuraciones]

    fig = plt.figure(figsize=(20,7),dpi=80)
    ax1 = fig.add_axes([0.02,0.12,0.43,0.75],projection="3d")
    ax2 = fig.add_axes([0.54,0.20,0.19,0.58])
    ax3 = fig.add_axes([0.79,0.20,0.19,0.58])

    def actualizar(frame):
        ax1.cla()
        X = superficies[frame]

        # SUPERFICIE 3D: cubos desde la base (z=0) hasta la altura h_i, con la cámara girando
        alturas = X.ravel()
        colores = cmap(norm(alturas))

        ax1.bar3d(j.ravel()-dx/2,
                  i.ravel()-dy/2,
                  np.zeros_like(alturas),
                  dx,dy,alturas,
                  color=colores,
                  edgecolor="black",
                  linewidth=0.2)

        ax1.set_title(f"Superficie - Iteración {iteraciones[frame]}",fontsize=13)
        ax1.set_xlim(-0.5,N-0.5)
        ax1.set_ylim(-0.5,N-0.5)
        ax1.set_zlim(min(0,zmin),zmax)   # incluye la base z=0 de los cubos
        ax1.set_xticks([])
        ax1.set_yticks([])
        ax1.set_zticks([])

        ax1.view_init(elev=25,azim=-65+120*frame/(len(superficies)-1))   # la cámara gira durante la animación
        ax1.set_box_aspect((N,N,12),zoom=1.2)
        ax1.xaxis.pane.fill = False
        ax1.yaxis.pane.fill = False
        ax1.zaxis.pane.fill = False
        ax1.grid(False)

        # ENERGÍA VS ITERACIONES
        ax2.cla()
        ax2.plot(iteraciones[:frame+1],energias[:frame+1],
                 color="royalblue",linewidth=2)
        ax2.scatter(iteraciones[frame],energias[frame],
                    color="red",s=60,zorder=5)

        ax2.set_xlim(0,n)
        margen = max(0.1,0.05*(max(energias)-min(energias)))
        ax2.set_ylim(min(energias)-margen,max(energias)+margen)
        ax2.set_xlabel("Iteraciones")
        ax2.set_ylabel("Energía de Dirichlet")
        ax2.set_title(f"Energía = {energias[frame]:.4f}")
        ax2.grid(alpha=0.3)

        # BETA VS ITERACIONES
        ax3.cla()
        ax3.plot(iteraciones[:frame+1],historial_beta[:frame+1],
                 color="darkorange",linewidth=2)
        ax3.scatter(iteraciones[frame],historial_beta[frame],
                    color="red",s=60,zorder=5)

        ax3.set_xlim(0,n)
        ax3.set_ylim(0,betaf*1.05)
        ax3.set_xlabel("Iteraciones")
        ax3.set_ylabel("β")
        ax3.set_title(f"β actual = {historial_beta[frame]:.4f}")
        ax3.grid(alpha=0.3)

    # BARRA DE COLORES
    sm = plt.cm.ScalarMappable(cmap=cmap,norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm,ax=ax1,shrink=0.55,pad=0.08,aspect=25)
    cbar.set_label("Altura",fontsize=11)

    # ANIMACIÓN
    fig.suptitle(f"Dinámica de Glauber - {nombre}",fontsize=16)
    anim = FuncAnimation(fig,actualizar,frames=len(superficies),interval=150,repeat=False)

    plt.close(fig)
    return anim

# MOSTRAMOS LAS CINCO ANIMACIONES
animaciones = {}

for nombre,(configuraciones,iteraciones,energias,historial_beta) in resultados.items():
    animaciones[nombre] = CrearAnimacion(nombre,configuraciones,iteraciones,energias,historial_beta)
    display(HTML(animaciones[nombre].to_jshtml()))

### Animación HeatMap

In [ ]:
### Animación plana (heat map) para los 5 métodos
# Usa los resultados, `a`, `norm`, `cmap`, `n` y `betaf` de la celda anterior (no hay que volver a simular).

def CrearAnimacionPlana(resultados,a,norm,cmap,n,intervalo=150):
    nombres = list(resultados.keys())
    nf = min(len(resultados[nom][1]) for nom in nombres)       # frames disponibles
    superficies = {nom:[a+phi for phi in resultados[nom][0][:nf]] for nom in nombres}
    iteraciones = resultados[nombres[0]][1][:nf]
    colores_met = {"Fijo":"tab:blue","Lineal":"tab:orange","Cuadrático":"tab:purple",
                   "Geométrico":"tab:green","Calentamiento-Enfriamiento":"tab:red"}

    fig = plt.figure(figsize=(19,8),dpi=80)
    gs = fig.add_gridspec(2,len(nombres),height_ratios=[1.25,1],hspace=0.35,wspace=0.12,
                          left=0.07,right=0.93,top=0.88,bottom=0.09)

    # HEAT MAPS (fila superior): h = a + phi, misma escala de color que los cubos 3D
    imagenes = {}
    for col,nom in enumerate(nombres):
        ax = fig.add_subplot(gs[0,col])
        imagenes[nom] = ax.imshow(superficies[nom][0],cmap=cmap,norm=norm,origin="lower")
        ax.set_xticks([]); ax.set_yticks([])
        imagenes[nom].axes_title = ax.set_title("",fontsize=10)

    cax = fig.add_axes([0.945,0.52,0.012,0.33])
    fig.colorbar(plt.cm.ScalarMappable(cmap=cmap,norm=norm),cax=cax).set_label("Altura")

    # ENERGÍA Y BETA (fila inferior) para los cinco métodos
    inferior = gs[1,:].subgridspec(1,2,wspace=0.15)
    ax_e = fig.add_subplot(inferior[0])
    ax_b = fig.add_subplot(inferior[1])
    lineas_e, puntos_e, lineas_b, puntos_b = {}, {}, {}, {}
    todas_e = np.concatenate([resultados[nom][2][:nf] for nom in nombres])
    for nom in nombres:
        lineas_e[nom], = ax_e.plot([],[],color=colores_met.get(nom,"gray"),lw=2,label=nom)
        puntos_e[nom], = ax_e.plot([],[],"o",color=colores_met.get(nom,"gray"))
        lineas_b[nom], = ax_b.plot([],[],color=colores_met.get(nom,"gray"),lw=2,label=nom)
        puntos_b[nom], = ax_b.plot([],[],"o",color=colores_met.get(nom,"gray"))
    ax_e.set_xlim(0,n); ax_e.set_yscale("log")
    ax_e.set_ylim(todas_e.min()*0.9,todas_e.max()*1.1)
    ax_e.set_xlabel("Iteraciones"); ax_e.set_ylabel("Energía de Dirichlet"); ax_e.grid(alpha=0.3)
    ax_e.legend(fontsize=9,loc="upper right")
    ax_b.set_xlim(0,n); ax_b.set_ylim(0,betaf*1.05)
    ax_b.set_xlabel("Iteraciones"); ax_b.set_ylabel("β"); ax_b.grid(alpha=0.3)

    titulo = fig.suptitle("",fontsize=15)

    def actualizar(frame):
        for nom in nombres:
            _,its,ens,bts = resultados[nom]
            imagenes[nom].set_data(superficies[nom][frame])
            imagenes[nom].axes_title.set_text(f"{nom}\nβ = {bts[frame]:.2f}   E = {ens[frame]:.2f}")
            lineas_e[nom].set_data(its[:frame+1],ens[:frame+1])
            puntos_e[nom].set_data([its[frame]],[ens[frame]])
            lineas_b[nom].set_data(its[:frame+1],bts[:frame+1])
            puntos_b[nom].set_data([its[frame]],[bts[frame]])
        titulo.set_text(f"Dinámica de Glauber (mapa de alturas h = a + φ) - Iteración {iteraciones[frame]:,}")

    anim = FuncAnimation(fig,actualizar,frames=nf,interval=intervalo,repeat=False)
    plt.close(fig)
    return anim

animacion_plana = CrearAnimacionPlana(resultados,a,norm,cmap,n)
display(HTML(animacion_plana.to_jshtml()))


### Animación hasta obtener la pirámide

In [ ]:
# PARÁMETROS
N = 40
beta = 11
n = 100000000
frames = 60
paso = 10000

# GENERAMOS LA CONFIGURACIÓN INICIAL Y LA PIRÁMIDE ÓPTIMA
i,j = np.indices((N,N))
d = np.minimum.reduce([i,j,N-1-i,N-1-j])

a = (d%3)/3
phi_optima = d//3
piramide = a+phi_optima

energia_optima = EnergiaTotal(a,phi_optima)

# GLAUBER FIJO HASTA ENCONTRAR EL ÓPTIMO
def IsingGlauberPiramideAnimacion(N,beta,a,n,energia_optima,paso):

    phi = np.zeros((N,N),dtype=int)
    energia_actual = EnergiaTotal(a,phi)
    mejor_energia = energia_actual
    mejor_phi = phi.copy()
    mejor_iteracion = 0
    configuraciones = [phi.copy()]
    iteraciones = [0]
    energias = [energia_actual]
    mejores_energias = [mejor_energia]
    alcanzo_optimo = np.isclose(energia_actual,energia_optima,atol=1e-8,rtol=0)
    iteracion_objetivo = 0 if alcanzo_optimo else None
    ultima_iteracion = 0

    for k in range(n):
        if alcanzo_optimo:
            break

        i = np.random.randint(0,N)
        j = np.random.randint(0,N)
        u = np.random.choice([-1,1])

        deltaH = DeltaEnergia(a,phi,i,j,u)
        proba = expit(-beta*deltaH)

        if np.random.rand()<proba:
            phi[i,j] += u
            energia_actual += deltaH

            # GUARDAMOS LA MEJOR CONFIGURACIÓN
            if energia_actual<mejor_energia:
                mejor_energia = energia_actual
                mejor_phi = phi.copy()
                mejor_iteracion = k+1

            # DETENEMOS GLAUBER AL ALCANZAR EL ÓPTIMO
            if energia_actual<=energia_optima+1e-8:
                if np.isclose(EnergiaTotal(a,phi),energia_optima,atol=1e-8,rtol=0):
                    alcanzo_optimo = True
                    iteracion_objetivo = k+1

        ultima_iteracion = k+1

        # GUARDAMOS CONFIGURACIONES INTERMEDIAS
        if (k+1)%paso==0 or alcanzo_optimo:
            configuraciones.append(phi.copy())
            iteraciones.append(k+1)
            energias.append(energia_actual)
            mejores_energias.append(mejor_energia)

    # GUARDAMOS LA CONFIGURACIÓN FINAL
    if iteraciones[-1]!=ultima_iteracion:
        configuraciones.append(phi.copy())
        iteraciones.append(ultima_iteracion)
        energias.append(energia_actual)
        mejores_energias.append(mejor_energia)

    return (configuraciones,iteraciones,energias,mejores_energias,mejor_phi,mejor_energia,mejor_iteracion,alcanzo_optimo,iteracion_objetivo)


# EJECUTAMOS GLAUBER FIJO
(configuraciones,iteraciones,energias,mejores_energias,mejor_phi,mejor_energia,mejor_iteracion,alcanzo_optimo,iteracion_objetivo) = IsingGlauberPiramideAnimacion(N,beta,a,n,energia_optima,paso)

# RESULTADOS
print("Energía óptima global:",round(energia_optima,6))
print("Mejor energía encontrada:",round(mejor_energia,6))
print("Iteración del mejor resultado:",mejor_iteracion)

if alcanzo_optimo:
    print("¡Se alcanzó el óptimo global!")
    print("Iteraciones necesarias:",iteracion_objetivo)
else:
    print("No se alcanzó el óptimo en",n,"iteraciones")

# SELECCIONAMOS 60 FRAMES DISTRIBUIDOS EN LA SIMULACIÓN
indices = np.unique(
    np.linspace(0,len(configuraciones)-1,min(frames,len(configuraciones)),dtype=int))

configuraciones = [configuraciones[k] for k in indices]
iteraciones = [iteraciones[k] for k in indices]
energias = [energias[k] for k in indices]
mejores_energias = [mejores_energias[k] for k in indices]

print("Frames de la animación:",len(configuraciones))

# ESCALA COMÚN
superficies = [a+phi for phi in configuraciones]

zmin = min(np.min(X) for X in superficies)-0.3
zmax = max(np.max(X) for X in superficies)+0.3

norm = plt.Normalize(vmin=zmin,vmax=zmax)
cmap = plt.cm.turbo

# POSICIONES DE LOS BLOQUES
i,j = np.indices((N,N))
dx = 0.9
dy = 0.9
grosor = 0.12

# ANIMACIÓN
def CrearAnimacionPiramide():

    fig = plt.figure(figsize=(20,7),dpi=70)

    ax1 = fig.add_axes([0.02,0.12,0.43,0.75],projection="3d")
    ax2 = fig.add_axes([0.54,0.20,0.19,0.58])
    ax3 = fig.add_axes([0.79,0.20,0.19,0.58])

    limite_iteraciones = iteraciones[-1]

    def actualizar(frame):

        ax1.cla()
        X = superficies[frame]

        # SUPERFICIE 3D
        alturas = X.ravel()
        colores = cmap(norm(alturas))
        ax1.bar3d(j.ravel()-dx/2,i.ravel()-dy/2,alturas-grosor/2,dx,dy,np.full_like(alturas,grosor),color=colores,edgecolor="black",linewidth=0.1)
        ax1.set_title(f"Superficie - Iteración {iteraciones[frame]:,}",fontsize=13)
        ax1.set_xlim(-0.5,N-0.5)
        ax1.set_ylim(-0.5,N-0.5)
        ax1.set_zlim(zmin,zmax)
        ax1.set_xticks([])
        ax1.set_yticks([])
        ax1.set_zticks([])
        ax1.view_init(elev=25,azim=-65+120*frame/max(1,len(superficies)-1))
        ax1.set_box_aspect((N,N,12),zoom=1.2)
        ax1.xaxis.pane.fill = False
        ax1.yaxis.pane.fill = False
        ax1.zaxis.pane.fill = False
        ax1.grid(False)

        # ENERGÍA VS ITERACIONES
        ax2.cla()
        ax2.plot(iteraciones[:frame+1],energias[:frame+1],color="royalblue",linewidth=2,label="Energía actual")
        ax2.plot(iteraciones[:frame+1],mejores_energias[:frame+1],color="green",linewidth=2,label="Mejor energía")
        ax2.axhline(energia_optima,color="red",linestyle="--",label="Óptimo global")
        ax2.scatter(iteraciones[frame],energias[frame],color="red",s=60)
        ax2.set_xlim(0,max(1,limite_iteraciones))
        margen = max(0.1,0.05*(max(energias)-energia_optima))
        ax2.set_ylim(energia_optima-margen,max(energias)+margen)
        ax2.set_xlabel("Iteraciones")
        ax2.set_ylabel("Energía de Dirichlet")
        ax2.set_title(f"Mejor energía = {mejores_energias[frame]:.4f}")
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.3)

        # BETA VS ITERACIONES
        ax3.cla()
        ax3.plot(iteraciones[:frame+1],[beta]*(frame+1),color="darkorange",linewidth=2)
        ax3.scatter(iteraciones[frame],beta,color="red",s=60)
        ax3.set_xlim(0,max(1,limite_iteraciones))
        ax3.set_ylim(0,beta*1.2)
        ax3.set_xlabel("Iteraciones")
        ax3.set_ylabel("β")
        ax3.set_title(f"β fijo = {beta}")
        ax3.grid(alpha=0.3)

        # MENSAJE AL ALCANZAR EL ÓPTIMO
        if alcanzo_optimo and frame==len(superficies)-1:
            ax1.set_title(f"¡ÓPTIMO ALCANZADO!\nIteración {iteracion_objetivo:,}",fontsize=13)

    # BARRA DE COLORES
    sm = plt.cm.ScalarMappable(cmap=cmap,norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm,ax=ax1,shrink=0.55,pad=0.08,aspect=25)
    cbar.set_label("Altura",fontsize=11)
    fig.suptitle("Glauber fijo - Búsqueda del óptimo global",fontsize=16)
    anim = FuncAnimation(fig,actualizar,frames=len(superficies),interval=150,repeat=False)
    plt.close(fig)
    return anim


# MOSTRAMOS LA ANIMACIÓN
anim = CrearAnimacionPiramide()
display(HTML(anim.to_jshtml()))

# Glauber vs. optimización con autograd

- los cinco métodos de Glauber
- una optimización de $\phi$ con **autograd de PyTorch** (φ continuo + penalidad que lo empuja a enteros, y redondeo final).

Métrica: **energía de Dirichlet por arista**, $E/(2N(N-1))$, para poder comparar entre tamaños.

Aquí β cambia **por barrido** (todas las casillas una vez) y no por cada actualización individual, y el calentamiento usa el 10 % inicial.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    HAY_TORCH = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    HAY_TORCH = False
    device = None
print("PyTorch:", HAY_TORCH, "| device:", device)

## Funciones básicas

In [ ]:
def energia(X):
    # Energía de Dirichlet total de X = a + phi
    return 0.5*np.sum((X[1:,:]-X[:-1,:])**2) + 0.5*np.sum((X[:,1:]-X[:,:-1])**2)

def aristas(N):
    return 2*N*(N-1)

def generar_a(N, rng):
    a = rng.uniform(0,1,size=(N,N))
    a[0,:] = 0; a[-1,:] = 0; a[:,0] = 0; a[:,-1] = 0
    return a

## Glauber vectorizado (tablero de ajedrez)

Las casillas pares no son vecinas entre sí (igual las impares), así que se pueden actualizar todas a la vez. Un **barrido** = actualizar las pares y luego las impares, es decir, cada casilla una vez en promedio, equivalente a $N^2$ actualizaciones de tu versión secuencial.

In [ ]:
def vecinos(X):
    S = np.zeros_like(X); Cn = np.zeros_like(X)
    S[1:,:] += X[:-1,:]; Cn[1:,:] += 1
    S[:-1,:] += X[1:,:]; Cn[:-1,:] += 1
    S[:,1:] += X[:,:-1]; Cn[:,1:] += 1
    S[:,:-1] += X[:,1:]; Cn[:,:-1] += 1
    return S, Cn

def glauber_tablero(a, betas, rng):
    # betas: un valor de beta por barrido
    N = a.shape[0]
    phi = np.zeros((N,N))
    ii, jj = np.indices((N,N))
    masks = [((ii+jj) % 2 == p) for p in (0,1)]
    for beta in betas:
        for m in masks:
            X = a + phi
            S, Cn = vecinos(X)
            u = rng.choice([-1,1], size=(N,N))
            dH = u*(Cn*X - S) + 0.5*Cn*u**2
            p = 1/(1 + np.exp(np.clip(beta*dH, -50, 50)))
            phi += u*((rng.random((N,N)) <= p) & m)
    return phi

def calendario(nombre, S, beta_fijo=11, beta0=1, betamin=0.5, betaf=20):
    k = np.arange(S)
    t = k/(S-1)
    if nombre == "Fijo":
        return np.full(S, float(beta_fijo))
    if nombre == "Lineal":
        return beta0 + (betaf-beta0)*t
    if nombre == "Cuadrático":
        return beta0 + (betaf-beta0)*t**2
    if nombre == "Geométrico":
        return beta0*(betaf/beta0)**t
    if nombre == "Calentamiento-Enfriamiento":
        cal = int(0.1*S)
        b = np.empty(S)
        b[:cal] = beta0*(betamin/beta0)**(k[:cal]/(cal-1))
        b[cal:] = betamin*(betaf/betamin)**((k[cal:]-cal)/(S-1-cal))
        return b
    raise ValueError(nombre)

METODOS_GLAUBER = ["Fijo","Lineal","Cuadrático","Geométrico","Calentamiento-Enfriamiento"]

## Optimización con autograd (PyTorch)

Se trata $\phi$ como parámetro **continuo** y se minimiza

$$E(a+\phi) + \lambda_t \sum_{ij} \sin^2(\pi\,\phi_{ij}),$$

con $\lambda_t$ creciendo (cuadráticamente) de 0 a `lam_max`; al final se redondea $\phi$ a enteros.

Con `lr=0.05` y `lam_max=1` es estable. **No subas `lam_max` a 10**: en mis pruebas en numpy se volvió inestable.

In [ ]:
if HAY_TORCH:
    def energia_total_torch(X):                 # X: (B,1,N,N)
        dx = X[:,:,1:,:] - X[:,:,:-1,:]
        dy = X[:,:,:,1:] - X[:,:,:,:-1]
        return 0.5*(dx**2).sum(dim=(1,2,3)) + 0.5*(dy**2).sum(dim=(1,2,3))

    def optimizar_phi(a_np, T=4000, lr=0.05, lam_max=1.0):
        a = torch.tensor(a_np, dtype=torch.float32, device=device)[None, None]
        phi = torch.zeros_like(a, requires_grad=True)
        for t in range(T):
            lam = lam_max*(t/T)**2
            loss = energia_total_torch(a+phi).sum() + lam*torch.sum(torch.sin(torch.pi*phi)**2)
            g, = torch.autograd.grad(loss, phi)
            with torch.no_grad():
                phi -= lr*g
        return torch.round(phi.detach())[0,0].cpu().numpy().astype(np.float64)

## Prueba rápida (N = 100)

In [ ]:
rng = np.random.default_rng(217896193)
N = 100
a = generar_a(N, rng)
E = aristas(N)
print("phi = 0           :", round(energia(a)/E, 4))

t0 = time.time()
phi = glauber_tablero(a, calendario("Geométrico", 250), rng)
print("Glauber geométrico:", round(energia(a+phi)/E, 4), f"({time.time()-t0:.1f}s)")

if HAY_TORCH:
    t0 = time.time()
    phi = optimizar_phi(a, T=4000)
    print("Autograd          :", round(energia(a+phi)/E, 4), f"({time.time()-t0:.1f}s)")

## Experimento: comparación para distintos N

- `S` = barridos por casilla, **igual para todos los N** (así $n = S\cdot N^2$ crece con el tamaño; `S=250` equivale a tu `n=100000` en N=20).
- Cada instancia usa la misma matriz `a` para todos los métodos.
- N=1000 con 6 métodos puede tardar varios minutos por instancia (la parte de autograd es mucho más rápida con GPU). Para una primera corrida, quita 500 y 1000 de `Ns`.

In [ ]:
# PARÁMETROS
Ns = [20, 50, 100, 200, 500, 1000]
R_por_N = {20:10, 50:10, 100:5, 200:3, 500:2, 1000:1}   # instancias por N
S = 250          # barridos por casilla
T_auto = 4000    # pasos de la optimización con autograd
semilla = 217896193

resultados = []
rng = np.random.default_rng(semilla)

for N in Ns:
    E = aristas(N)
    for r in range(R_por_N[N]):
        a = generar_a(N, rng)
        resultados.append((N, r, "φ = 0", energia(a)/E, 0.0))

        for nombre in METODOS_GLAUBER:
            t0 = time.time()
            phi = glauber_tablero(a, calendario(nombre, S), rng)
            resultados.append((N, r, nombre, energia(a+phi)/E, time.time()-t0))

        if HAY_TORCH:
            t0 = time.time()
            phi = optimizar_phi(a, T=T_auto)
            resultados.append((N, r, "Autograd", energia(a+phi)/E, time.time()-t0))
    print(f"N = {N} listo")

df = pd.DataFrame(resultados, columns=["N","instancia","Método","Energía por arista","Tiempo (s)"])

## Tablas

In [ ]:
orden = ["φ = 0"] + METODOS_GLAUBER + (["Autograd"] if HAY_TORCH else [])

tabla_E = df.pivot_table(index="N", columns="Método", values="Energía por arista", aggfunc="mean")[orden]
tabla_T = df[df["Método"]!="φ = 0"].pivot_table(index="N", columns="Método", values="Tiempo (s)", aggfunc="mean")[orden[1:]]

print("Energía por arista (promedio sobre instancias):")
print(tabla_E.to_string(float_format="%.4f"))
print("\nMejor método por N:")
print(tabla_E.drop(columns="φ = 0").idxmin(axis=1).to_string())
print("\nTiempo por corrida (s):")
print(tabla_T.to_string(float_format="%.1f"))

## Gráficos

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,5))

for m in orden:
    ax[0].plot(tabla_E.index, tabla_E[m], marker="o", label=m, ls="--" if m=="φ = 0" else "-")
ax[0].set_xscale("log")
ax[0].set_xlabel("N"); ax[0].set_ylabel("Energía de Dirichlet por arista")
ax[0].set_title("Calidad vs N"); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8)

for m in orden[1:]:
    ax[1].plot(tabla_T.index, tabla_T[m], marker="o", label=m)
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("N"); ax[1].set_ylabel("Tiempo (s)")
ax[1].set_title("Tiempo vs N"); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()